# geometric covariance pooling and spectral cleaning

This experiment tests a more expressive extension with **forecast accuracy and portfolio risk given equal importance**. It combines five covariance experts, a small regime-dependent gate, learned eigenvalue cleaning and a deterministic condition-number guard. Positive definiteness and the condition limit are mathematical properties. Better forecasts and lower realized risk require an empirical test.

[Design](docs/spectral/design.json) · [Architecture and sources](docs/spectral/research.md) · [Implementation](spectral.py)

The implementation is a project-specific combination inspired by geometric covariance forecasting and neural covariance cleaning. It is not an exact reproduction of those papers. The earlier 2025–2026 outputs have already been examined; this period is now a **reused retrospective benchmark**, not an untouched confirmation sample. No architecture or hyperparameters are selected by its results in this experiment.

![geometric spectral architecture](docs/spectral-architecture.png)

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from threadpoolctl import threadpool_limits

from market_data import load_market
from models import BASELINE, GRAPH_MODEL, Forecasts, forecast_grid
from evaluation import (
    bootstrap_means, covariance_losses, forecast_table, long_only_weights,
    point_ratios, portfolio_table,
)
from extension import ADAPTIVE_MODEL, UNIFORM_MODEL
from spectral import (
    DTYPE, SPECTRAL_MODEL, STATIC_MODEL, GeometricSpectralGate,
    differentiable_weights, spectral_forecasts,
)

config = json.loads(Path("docs/spectral/design.json").read_text())
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
table_directory = Path("docs/spectral/tables")
table_directory.mkdir(parents=True, exist_ok=True)
display(pd.Series(config))
model = GeometricSpectralGate(10, 5)
print(f"dynamic architecture: {sum(p.numel() for p in model.parameters())} trainable parameters")

architecture                    geometric covariance pool with regime gate and...
primary_endpoints               [one-session mean Frobenius distance, one-sess...
primary_reference                                                         HAR-DRD
strong_controls                 [Adaptive shape, Uniform shape, HAR-Cholesky, ...
expert_bank                     [HAR-DRD, GHAR(GL,L), HAR-Cholesky, arithmetic...
training                        equal normalized forecast-distance and squared...
rolling_history                                                               504
minimum_history                                                               378
inner_validation                                                              126
epoch_checkpoints                                              [0, 5, 10, 20, 40]
selection                       minimize worse of validation Frobenius ratio a...
seed                                                                         2026
learning_rate   

dynamic architecture: 175 trainable parameters


## 1. architecture and equal objectives

The expert bank contains HAR-DRD, GHAR(GL,L), HAR-Cholesky, an arithmetic daily/weekly/monthly covariance pool and its log-Euclidean counterpart. Graph information enters through the GHAR expert. This is a gated expert architecture, rather than a newly trained graph-message-passing network.

The dynamic gate uses past trace ratios, covariance conditioning, volatility dispersion and average correlation. It learns five mixture weights and a bounded total-variance adjustment. A shared spectral network learns how much to shrink each pooled eigenvalue toward their mean. A final guard bounds the covariance condition number by 100. A static pool with scalar shrinkage is trained as a simpler control.

Training minimizes the equally weighted mean of two historically normalized losses: Frobenius distance and squared realized long-only portfolio return. Long-only weights during training use 40 accelerated projected-gradient steps. Evaluation uses exact constrained optimization; the approximation gap is reported separately.

Gate refits occur quarterly, while original HAR experts retain monthly refits and all daily inputs update causally. Each gate uses at most 504 historical labels. The last 126 form an inner validation block; checkpoint selection minimizes the worse of the two validation ratios against HAR-DRD. The selected epoch count is then refitted on all available historical labels. This validation rule does not guarantee an improvement on either endpoint.

In [2]:
with threadpool_limits(limits=1):
    panel = load_market("syncster", Path("data/syncster"), "2016-01-04", config["data_end"])
    base = forecast_grid(panel, horizons=(1,), lookback=1000)[1]
print(f"{len(panel.symbols)} stocks; {len(panel.dates)} retained sessions")

27 stocks; 2090 retained sessions


In [3]:
with threadpool_limits(limits=1):
    extension = spectral_forecasts(panel, base, config, progress=print)
result = extension.forecasts
audit = extension.audit
assert (pd.to_datetime(audit["last_refit_label"]) < pd.to_datetime(audit["first_forecast"])).all()
assert (pd.to_datetime(audit["last_fit_label"]) < pd.to_datetime(audit["first_validation_label"])).all()
audit.to_csv(table_directory / "fold-audit.csv", index=False)
extension.gate_weights.to_csv(table_directory / "gate-weights.csv", index_label="date")
display(audit)

2024Q1: historical labels 378; forecasts 57


2024Q2: historical labels 435; forecasts 63


2024Q3: historical labels 498; forecasts 63


2024Q4: historical labels 504; forecasts 62


2025Q1: historical labels 504; forecasts 60


2025Q2: historical labels 504; forecasts 62


... (3 outputs left out)

2026Q2: historical labels 504; forecasts 58


2026Q3: historical labels 504; forecasts 61


2026Q4: historical labels 504; forecasts 16


,period,model,first_forecast,last_fit_label,first_validation_label,last_validation_label,last_refit_label,historical_labels,chosen_epoch,validation_frobenius_ratio,validation_risk_proxy_ratio
0,2024Q1,Geometric spectral gate,2024-01-08,2023-07-06,2023-07-07,2024-01-05,2024-01-05,378,10,0.896661,0.955457
1,2024Q1,Static geometric pool,2024-01-08,2023-07-06,2023-07-07,2024-01-05,2024-01-05,378,20,0.896089,0.957017
2,2024Q2,Geometric spectral gate,2024-04-01,2023-09-26,2023-09-27,2024-03-28,2024-03-28,435,20,0.897123,0.960385
3,2024Q2,Static geometric pool,2024-04-01,2023-09-26,2023-09-27,2024-03-28,2024-03-28,435,40,0.899604,0.964476
4,2024Q3,Geometric spectral gate,2024-07-01,2023-12-27,2023-12-28,2024-06-28,2024-06-28,498,0,0.916900,0.991302
5,2024Q3,Static geometric pool,2024-07-01,2023-12-27,2023-12-28,2024-06-28,2024-06-28,498,0,0.916900,0.991302
6,2024Q4,Geometric spectral gate,2024-10-01,2024-03-28,2024-04-01,2024-09-30,2024-09-30,504,0,0.937969,1.013722
7,2024Q4,Static geometric pool,2024-10-01,2024-03-28,2024-04-01,2024-09-30,2024-09-30,504,0,0.937969,1.013722
8,2025Q1,Geometric spectral gate,2025-01-02,2024-06-27,2024-06-28,2024-12-31,2024-12-31,504,10,0.941340,1.006977
9,2025Q1,Static geometric pool,2025-01-02,2024-06-27,2024-06-28,2024-12-31,2024-12-31,504,20,0.944000,1.010111


## 2. forecast accuracy and portfolio risk

Every model is scored on the same dates. Development covers 2024; the reused benchmark starts in 2025. The primary candidate is the dynamic spectral gate. The static control and the earlier adaptive mixture are reported regardless of ranking.

In [4]:
cutoff = pd.Timestamp(config["reused_benchmark_start"])
periods = {}
loss_frames = {}
portfolio_frames = {}
weight_cache = {}
for period, mask in {
    "development": result.dates < cutoff,
    "reused benchmark": result.dates >= cutoff,
}.items():
    sample = Forecasts(
        result.dates[mask], result.positions[mask], result.actual[mask],
        {name: matrix[mask] for name, matrix in result.predictions.items()}, result.audit,
    )
    periods[period] = sample
    with threadpool_limits(limits=1):
        losses = covariance_losses(sample)
        table = forecast_table(losses, draws=10_000, block=config["bootstrap_mean_block"])
        portfolio = portfolio_table(sample, panel.portfolio_return)
        weight_cache[period] = {
            name: long_only_weights(matrix) for name, matrix in sample.predictions.items()
        }
    loss_frames[period], portfolio_frames[period] = losses, portfolio
    filename = period.replace(" ", "-")
    table.to_csv(table_directory / f"{filename}-matrix.csv")
    portfolio.to_csv(table_directory / f"{filename}-portfolio.csv", index=False)
    print(f"{period}: {len(sample.dates)} shared origins")
    display(table.round(4))
    display(portfolio.round(6))

development: 245 shared origins


Euclidean              Frobenius               \
                                 ratio rank   MCS p     ratio rank   MCS p   
model                                                                        
HAR-Cholesky                    0.9467    4  0.0881    0.9445    4  0.0635   
HAR-DRD                         1.0000    9  0.0001    1.0000    9  0.0001   
GHAR(GL,L)                      0.9757    5  0.0003    0.9765    5  0.0002   
Uniform shape                   0.9865    7  0.0001    0.9885    8  0.0001   
Adaptive shape                  0.9857    6  0.0001    0.9875    7  0.0001   
Arithmetic HAR covariance       0.9870    8  0.0028    0.9818    6  0.0013   
Log-Euclidean HAR covariance    0.9395    3  0.5543    0.9362    3  0.7411   
Geometric spectral gate         0.9359    2  0.5543    0.9335    2  0.7411   
Static geometric pool           0.9344    1  1.0000    0.9326    1  1.0000   

                               QLIKE               
                               ratio rank   MCS p  
model                                              
HAR-Cholesky                  1.0807    8  0.0001  
HAR-DRD                       1.0000    4  0.0001  
GHAR(GL,L)                    0.9852    1  1.0000  
Uniform shape                 0.9898    3  0.0001  
Adaptive shape                0.9891    2  0.0055  
Arithmetic HAR covariance     1.0348    7  0.0001  
Log-Euclidean HAR covariance  1.1307    9  0.0001  
Geometric spectral gate       1.0090    6  0.0001  
Static geometric pool         1.0049    5  0.0001

,model,portfolio,annual risk (%),turnover
0,1/N,equal,8.587514,0.007810
1,HAR-Cholesky,GMVP,7.711117,0.472918
2,HAR-Cholesky,GMVP+,7.561367,0.333173
3,HAR-DRD,GMVP,7.552072,0.540131
4,HAR-DRD,GMVP+,7.426245,0.386573
5,"GHAR(GL,L)",GMVP,7.454628,0.511270
6,"GHAR(GL,L)",GMVP+,7.438704,0.390464
7,Uniform shape,GMVP,7.524667,0.473242
8,Uniform shape,GMVP+,7.433382,0.351835
9,Adaptive shape,GMVP,7.514118,0.474099


reused benchmark: 443 shared origins


Euclidean              Frobenius               \
                                 ratio rank   MCS p     ratio rank   MCS p   
model                                                                        
HAR-Cholesky                    1.0137    8  0.0984    1.0062    7  0.1114   
HAR-DRD                         1.0000    6  0.1633    1.0000    6  0.1114   
GHAR(GL,L)                      1.0113    7  0.1857    1.0109    8  0.1410   
Uniform shape                   0.9982    5  0.1952    0.9987    5  0.1410   
Adaptive shape                  0.9977    4  0.1906    0.9979    4  0.1664   
Arithmetic HAR covariance       1.0827    9  0.0869    1.0906    9  0.0940   
Log-Euclidean HAR covariance    0.9860    3  0.2621    0.9808    2  0.3277   
Geometric spectral gate         0.9839    2  0.2621    0.9821    3  0.3219   
Static geometric pool           0.9766    1  1.0000    0.9740    1  1.0000   

                               QLIKE               
                               ratio rank   MCS p  
model                                              
HAR-Cholesky                  1.0764    7  0.0001  
HAR-DRD                       1.0000    5  0.0001  
GHAR(GL,L)                    1.1609    9  0.0001  
Uniform shape                 0.9926    2  0.0001  
Adaptive shape                0.9918    1  1.0000  
Arithmetic HAR covariance     1.0158    6  0.0001  
Log-Euclidean HAR covariance  1.0897    8  0.0001  
Geometric spectral gate       0.9961    3  0.0001  
Static geometric pool         0.9999    4  0.0001

,model,portfolio,annual risk (%),turnover
0,1/N,equal,12.596146,0.010125
1,HAR-Cholesky,GMVP,9.169764,0.436514
2,HAR-Cholesky,GMVP+,9.404682,0.307518
3,HAR-DRD,GMVP,9.325832,0.457929
4,HAR-DRD,GMVP+,9.435110,0.328546
5,"GHAR(GL,L)",GMVP,30.161618,0.770450
6,"GHAR(GL,L)",GMVP+,9.746767,0.336007
7,Uniform shape,GMVP,9.221239,0.414509
8,Uniform shape,GMVP+,9.466393,0.304874
9,Adaptive shape,GMVP,9.227863,0.415365


## 3. joint success rule and paired uncertainty

The predefined joint success rule requires both primary paired 95% intervals against HAR-DRD to lie strictly below zero: mean Frobenius distance and long-only realized portfolio variance. Both outcomes must improve; one cannot compensate for the other. The two intervals use the same dates and 10,000 seeded stationary-bootstrap draws with mean block length 10.

Portfolio intervals resample return means and second moments jointly and recenter each bootstrap replicate. Additional comparisons against the earlier adaptive and uniform mixtures, HAR-Cholesky and the static pool are exploratory. Testing more controls does not turn this reused benchmark into an independent confirmation.

In [5]:
records = []
for period, sample in periods.items():
    daily = loss_frames[period].pivot(index="date", columns="model", values="Frobenius")
    actual_returns = panel.portfolio_return[sample.positions]
    candidate_returns = (weight_cache[period][SPECTRAL_MODEL] * actual_returns).sum(axis=1)
    for reference in [BASELINE, ADAPTIVE_MODEL, UNIFORM_MODEL, "HAR-Cholesky", STATIC_MODEL]:
        reference_returns = (weight_cache[period][reference] * actual_returns).sum(axis=1)
        data = np.column_stack([
            daily[SPECTRAL_MODEL], daily[reference], candidate_returns, reference_returns,
            candidate_returns ** 2, reference_returns ** 2,
        ])
        samples = bootstrap_means(
            data, config["bootstrap_draws"], config["bootstrap_mean_block"],
            np.random.default_rng(config["seed"]),
        )
        relative_distance = samples[:, 0] / samples[:, 1] - 1
        relative_variance = (samples[:, 4] - samples[:, 2] ** 2) / (
            samples[:, 5] - samples[:, 3] ** 2
        ) - 1
        mean = data.mean(axis=0)
        distance_interval = np.quantile(relative_distance, [.025, .975])
        risk_interval = np.quantile(relative_variance, [.025, .975])
        records.append({
            "period": period, "reference": reference,
            "Frobenius change (%)": 100 * (mean[0] / mean[1] - 1),
            "Frobenius 95% lower": 100 * distance_interval[0],
            "Frobenius 95% upper": 100 * distance_interval[1],
            "long-only variance change (%)": 100 * (
                (mean[4] - mean[2] ** 2) / (mean[5] - mean[3] ** 2) - 1
            ),
            "variance 95% lower": 100 * risk_interval[0],
            "variance 95% upper": 100 * risk_interval[1],
            "both intervals below zero": bool(distance_interval[1] < 0 and risk_interval[1] < 0),
        })
paired_results = pd.DataFrame(records).set_index(["period", "reference"])
paired_results.to_csv(table_directory / "paired-comparisons.csv")
display(paired_results.round(4))
primary = paired_results.loc[("reused benchmark", BASELINE)]
print("predefined joint benchmark criterion:", "passed" if primary["both intervals below zero"] else "not passed")

Frobenius change (%)  \
period           reference                                     
development      HAR-DRD                             -6.6464   
                 Adaptive shape                      -5.4655   
                 Uniform shape                       -5.5634   
                 HAR-Cholesky                        -1.1651   
                 Static geometric pool                0.0997   
reused benchmark HAR-DRD                             -1.7889   
                 Adaptive shape                      -1.5800   
                 Uniform shape                       -1.6636   
                 HAR-Cholesky                        -2.3961   
                 Static geometric pool                0.8330   

                                        Frobenius 95% lower  \
period           reference                                    
development      HAR-DRD                            -8.7879   
                 Adaptive shape                     -7.6181   
                 Uniform shape                      -7.7404   
                 HAR-Cholesky                       -2.2286   
                 Static geometric pool              -0.2495   
reused benchmark HAR-DRD                            -2.8223   
                 Adaptive shape                     -3.1139   
                 Uniform shape                      -3.2978   
                 HAR-Cholesky                       -4.1708   
                 Static geometric pool              -0.1297   

                                        Frobenius 95% upper  \
period           reference                                    
development      HAR-DRD                            -4.7450   
                 Adaptive shape                     -3.5515   
                 Uniform shape                      -3.6169   
                 HAR-Cholesky                        0.0272   
                 Static geometric pool               0.4709   
reused benchmark HAR-DRD                            -0.6606   
                 Adaptive shape                     -0.1895   
                 Uniform shape                      -0.2076   
                 HAR-Cholesky                        0.0260   
                 Static geometric pool               2.1910   

                                        long-only variance change (%)  \
period           reference                                              
development      HAR-DRD                                       0.6197   
                 Adaptive shape                                0.5553   
                 Uniform shape                                 0.4265   
                 HAR-Cholesky                                 -2.9444   
                 Static geometric pool                         0.1640   
reused benchmark HAR-DRD                                      -0.6131   
                 Adaptive shape                               -1.7840   
                 Uniform shape                                -1.2689   
                 HAR-Cholesky                                  0.0310   
                 Static geometric pool                        -1.1089   

                                        variance 95% lower  \
period           reference                                   
development      HAR-DRD                           -1.5719   
                 Adaptive shape                    -0.7423   
                 Uniform shape                     -0.9233   
                 HAR-Cholesky                      -6.1189   
                 Static geometric pool             -0.1351   
reused benchmark HAR-DRD                           -2.1492   
                 Adaptive shape                    -3.6196   
                 Uniform shape                     -2.6499   
                 HAR-Cholesky                      -3.3931   
                 Static geometric pool             -2.1592   

                                        variance 95% upper  \
period           reference                                   
development

predefined joint benchmark criterion: not passed


## 4. structural guarantees and solver approximation

Every bank forecast is positive definite, so a convex pool is positive definite. Each cleaned eigenvalue is a convex combination of a positive eigenvalue and their positive mean. Renormalization preserves trace, and a positive scale adjustment preserves definiteness.

For positive eigenvalues with maximum $M$, minimum $m$ and mean $\mu$, the guard chooses

$$a=\frac{(M-\kappa m)_+}{(M-\kappa m)_++(\kappa-1)\mu},\qquad
\lambda_i'=(1-a)\lambda_i+a\mu.$$

Direct substitution gives $\lambda_{\max}'/\lambda_{\min}'\le\kappa$. This is a deterministic numerical guarantee. For unrestricted fully invested GMVP weights, comparison with equal weights also yields $\|w\|_1\le\sqrt\kappa$: positive definiteness gives $m\|w\|_2^2\le w^\top Sw\le M/n$, and Cauchy–Schwarz completes the bound. At $\kappa=100$, gross leverage is at most 10. This is a worst-case bound, not a claim of optimal portfolio risk.

In [6]:
diagnostics = []
for period, sample in periods.items():
    predicted = sample.predictions[SPECTRAL_MODEL]
    eigenvalues = np.linalg.eigvalsh(predicted)
    assert (eigenvalues > 0).all()
    assert (eigenvalues[:, -1] / eigenvalues[:, 0] <= config["condition_limit"] + 1e-6).all()
    with torch.no_grad():
        approximate = differentiable_weights(
            torch.as_tensor(predicted, dtype=DTYPE), config["differentiable_simplex_steps"],
        ).numpy()
    exact = weight_cache[period][SPECTRAL_MODEL]
    exact_objective = np.einsum("ti,tij,tj->t", exact, predicted, exact)
    approximate_objective = np.einsum("ti,tij,tj->t", approximate, predicted, approximate)
    relative_gap = approximate_objective / exact_objective - 1
    diagnostics.append({
        "period": period,
        "maximum condition number": np.max(eigenvalues[:, -1] / eigenvalues[:, 0]),
        "mean training-solver relative variance gap": relative_gap.mean(),
        "maximum training-solver relative variance gap": relative_gap.max(),
        "maximum weights L1 difference": np.abs(approximate - exact).sum(axis=1).max(),
    })
solver_audit = pd.DataFrame(diagnostics).set_index("period")
solver_audit.to_csv(table_directory / "solver-audit.csv")
display(solver_audit.round(8))

,maximum condition number,mean training-solver relative variance gap,maximum training-solver relative variance gap,maximum weights L1 difference
period,,,,
development,69.518074,0.000006,0.000076,0.018251
reused benchmark,100.000000,0.000014,0.000199,0.035274


## 5. interactive plots

These charts show the fixed candidate alongside all simpler alternatives. The marker separates development from the reused benchmark.

In [7]:
weight_plot = extension.gate_weights.rename_axis("date").reset_index().melt(
    id_vars="date", var_name="expert", value_name="weight",
)
fig = px.line(weight_plot, x="date", y="weight", color="expert", title="geometric spectral gate · weights before each outcome")
fig.add_vline(x=cutoff.value / 1_000_000, line_dash="dot", line_color="#647880")
fig.update_layout(template="plotly_white", height=400)
fig.show(renderer="plotly_mimetype")
ratios = point_ratios(loss_frames["reused benchmark"]).rename_axis("model").reset_index().melt(
    id_vars="model", var_name="loss", value_name="ratio to har-drd",
)
fig = px.bar(ratios, x="model", y="ratio to har-drd", color="loss", barmode="group", title="reused benchmark · covariance forecast losses")
fig.add_hline(y=1, line_dash="dot", line_color="#647880")
fig.update_layout(template="plotly_white", height=450)
fig.show(renderer="plotly_mimetype")
portfolio = portfolio_frames["reused benchmark"]
fig = px.bar(portfolio[portfolio.portfolio.eq("GMVP+")], x="model", y="annual risk (%)", title="reused benchmark · long-only annualized gross risk")
fig.update_layout(template="plotly_white", height=420)
fig.show(renderer="plotly_mimetype")

## 6. interpretation

Read the paired comparison table before claiming an improvement. The joint criterion requires both endpoints to pass against HAR-DRD; the older adaptive model and static pool are stronger additional controls. The saved design fixes the candidate, equal training objective, checkpoint grid and condition limit before this run's benchmark evaluation.

The same data limitations apply as in the original public replication: the panel uses trade bars rather than the original licensed LOBSTER quote feed, omitted dates compress the retained-session calendar, and portfolio returns exclude transaction costs. The condition guarantee addresses matrix stability. Fresh dates or an independently chosen asset universe are still needed to confirm gains after this architecture search.

The saved run lowers Frobenius distance by **1.79%** and long-only annualized risk from **9.4351% to 9.4061%** against HAR-DRD. The paired risk interval includes zero, so the predefined joint criterion is **not passed**. The static pool has better matrix-distance point results, while the dynamic candidate has lower risk than that static control. [Full interpretation](docs/spectral/results.md).